# Notebook 02 — Stochastic analysis pipeline

**Paper:** *Stochastic collapse and first-passage dynamics in marathon pacing:
a Kramers–Moyal approach*  
**Author:** A. Domínguez-Monterroza  
**Target:** Physical Review E

This notebook implements the complete stochastic analysis pipeline from
preprocessed data (output of Notebook 01) to publication-quality figures.

| Section | Content |
|---------|----------|
| 1 | Imports and configuration |
| 2 | Load processed data |
| 3 | State variable and distance grid |
| 4 | Transition dataframe |
| 5 | Kramers–Moyal estimation with bootstrap CI |
| 6 | Pawula diagnostics |
| 7 | Markov diagnostics |
| 8 | First-passage statistics and hazard |
| 9 | Effective potential and bifurcation analysis |
| 10 | Langevin simulation — in-sample validation |
| 11 | Out-of-sample validation (Berlin Marathon) |
| 12 | Robustness checks |
| 13 | Publication figures (PRE style) |
| 14 | Diagnostic summary |

**Prerequisites:** run Notebook 01 first to generate
`data/processed/segment_paces_data.csv` and `data/processed/wall_labels_data.csv`.

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.interpolate import interp1d
from scipy.integrate import cumulative_trapezoid
from scipy.ndimage import gaussian_filter1d
from scipy.stats import ks_2samp, wasserstein_distance, gaussian_kde
from sklearn.metrics import r2_score

np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path('data/processed')
FIGURES_DIR = Path('figures')
RESULTS_DIR = Path('results')
for d in [FIGURES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Collapse definition (Paper §II.C) ─────────────────────────────────────────
Y_THRESHOLD    = 0.10   # primary threshold y_c
LATE_START_IDX = 5      # segment index for x >= 25 km

# ── Cross-validation split (Paper §VI.B) ─────────────────────────────────────
TRAIN_CITIES = ['Boston', 'Chicago', 'NewYork']
TEST_CITIES  = ['Berlin']

# ── KM estimation parameters (Paper §III.A) ──────────────────────────────────
KM_N_BINS    = 30    # state bins
KM_MIN_COUNT = 250   # minimum observations per bin
KM_N_BOOT    = 500   # bootstrap resamples
KM_CI        = 0.95  # confidence level

# ── Simulation parameters (Paper §VI.A) ──────────────────────────────────────
SIM_N_PATHS  = 30_000
SIM_CLIP_LOW = -0.5
SIM_CLIP_HIGH = 1.5

# ── Distance grid ─────────────────────────────────────────────────────────────
SEG_ORDER = [
    '0_5k', '5_10k', '10_15k', '15_20k', '20k_half',
    'half_25k', '25_30k', '30_35k', '35_40k', '40k_finish'
]
SEGMENT_END_KM = np.array(
    [5.0, 10.0, 15.0, 20.0, 21.0975, 25.0, 30.0, 35.0, 40.0, 42.195]
)
SEGMENT_MID_KM = np.array(
    [2.5, 7.5, 12.5, 17.5, 20.549, 23.049, 27.5, 32.5, 37.5, 41.098]
)
NORM_DIST = SEGMENT_END_KM / 42.195

print('Configuration OK')

In [ ]:
def set_pre_style() -> None:
    """Apply PRE-compliant matplotlib style (full frame, inward ticks)."""
    mpl.rcParams.update({
        'font.size':          14,
        'axes.labelsize':     15,
        'axes.titlesize':     14,
        'xtick.labelsize':    13,
        'ytick.labelsize':    13,
        'legend.fontsize':    12,
        'axes.spines.top':    True,
        'axes.spines.right':  True,
        'xtick.direction':    'in',
        'ytick.direction':    'in',
        'xtick.top':          True,
        'ytick.right':        True,
        'xtick.major.size':   5,
        'ytick.major.size':   5,
        'xtick.minor.size':   3,
        'ytick.minor.size':   3,
        'axes.linewidth':     1.2,
        'axes.grid':          False,
        'figure.dpi':         150,
        'savefig.dpi':        300,
        'savefig.bbox':       'tight',
    })

set_pre_style()
print('PRE style applied.')

## 2. Load processed data

In [ ]:
seg_df  = pd.read_csv(DATA_DIR / 'segment_paces_data.csv')
wall_df = pd.read_csv(DATA_DIR / 'wall_labels_data.csv')

merge_keys = [
    c for c in ['runner_id', 'City', 'Year', 'Sex', 'Age',
                'finish_time_min', 'performance_group']
    if c in seg_df.columns and c in wall_df.columns
]
df = seg_df.merge(wall_df, on=merge_keys, how='inner')

print(f'Total runners: {len(df):,}')
if 'City' in df.columns:
    print(df.groupby(['City', 'Year']).size().to_string())

## 3. State variable and distance grid

In [ ]:
# Primary state variable columns y^B (Paper §II.B)
state_cols = [f'obsB_{s}' for s in SEG_ORDER]

# Verify columns exist
missing = [c for c in state_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing state columns: {missing}')

# State matrix: shape (N, n_segments)
state_matrix = df[state_cols]

print(f'State columns: {state_cols}')
print(f'State matrix shape: {state_matrix.shape}')
print(f'\nState variable summary (obsB):')
print(state_matrix.describe().round(3))

## 4. Transition dataframe

In [ ]:
def build_transition_df(df: pd.DataFrame, state_cols: list,
                        norm_dist: np.ndarray,
                        segment_end_km: np.ndarray) -> pd.DataFrame:
    """
    Build long-format dataframe of segment-to-segment increments
    Delta y_{i,k} = y_{i,k+1} - y_{i,k} (Paper §III.A).

    Returns
    -------
    pd.DataFrame with columns: k, km_left, km_right, dx, y, dy
    """
    meta_cols = [
        c for c in ['runner_id', 'City', 'Year', 'Sex', 'Age',
                    'performance_group', 'finish_time_min',
                    'wall_persistent', 'wall_idx_persistent',
                    'wall_km_persistent']
        if c in df.columns
    ]
    records = []
    for k in range(len(state_cols) - 1):
        dx = norm_dist[k + 1] - norm_dist[k]
        block = df[meta_cols].copy()
        block['k']        = k
        block['km_left']  = segment_end_km[k]
        block['km_right'] = segment_end_km[k + 1]
        block['dx']       = dx
        block['y']        = df[state_cols[k]].values
        block['dy']       = df[state_cols[k + 1]].values - df[state_cols[k]].values
        records.append(block)

    long_df = pd.concat(records, ignore_index=True)
    return long_df.dropna(subset=['y', 'dy'])


long_df = build_transition_df(df, state_cols, NORM_DIST, SEGMENT_END_KM)
print(f'Transition dataframe shape: {long_df.shape}')

## 5. Kramers–Moyal estimation with bootstrap confidence intervals

Estimators (Paper §III.A, Eqs. 3–4):

$$D^{(1)}(y,x) \approx \frac{\langle\Delta y\mid y,x\rangle}{\Delta x}, \qquad
D^{(2)}(y,x) \approx \frac{\langle(\Delta y)^2\mid y,x\rangle}{2\,\Delta x}$$

Uncertainty is quantified via non-parametric bootstrap with $B=500$ resamples per bin.

In [ ]:
def estimate_km_bootstrap(
    long_df: pd.DataFrame,
    n_bins: int = 30,
    min_count: int = 250,
    n_boot: int = 500,
    ci: float = 0.95,
) -> pd.DataFrame:
    """
    Estimate D1 and D2 per (state bin, transition k) with bootstrap CI.

    Returns
    -------
    pd.DataFrame with columns:
        k, km_left, km_right, state_center, count,
        D1, D1_lo, D1_hi, D2, D2_lo, D2_hi
    """
    y_global = long_df['y'].values
    q_lo, q_hi = np.nanquantile(y_global, [0.01, 0.99])
    bins    = np.linspace(q_lo, q_hi, n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    alpha   = (1 - ci) / 2
    rng     = np.random.default_rng(seed=42)
    records = []

    for k, sub_k in long_df.groupby('k'):
        dx_ref = float(np.nanmedian(sub_k['dx'].values))
        sub_k  = sub_k.copy()
        sub_k['bin'] = pd.cut(sub_k['y'], bins=bins,
                              include_lowest=True, labels=False)
        for b, yc in enumerate(centers):
            dy = sub_k.loc[sub_k['bin'] == b, 'dy'].values
            n  = len(dy)
            if n < min_count:
                continue

            d1 = np.mean(dy) / dx_ref
            d2 = np.mean(dy ** 2) / (2 * dx_ref)

            boot_d1 = np.array([
                np.mean(rng.choice(dy, size=n, replace=True)) / dx_ref
                for _ in range(n_boot)
            ])
            boot_d2 = np.array([
                np.mean(rng.choice(dy, size=n, replace=True) ** 2) / (2 * dx_ref)
                for _ in range(n_boot)
            ])

            records.append({
                'k': k,
                'km': float(sub_k['km_right'].iloc[0]),
                'km_left':  float(sub_k['km_left'].iloc[0]),
                'km_right': float(sub_k['km_right'].iloc[0]),
                'state_center': yc,
                'count': n,
                'D1': d1, 'D1_lo': np.quantile(boot_d1, alpha),
                           'D1_hi': np.quantile(boot_d1, 1 - alpha),
                'D2': d2, 'D2_lo': np.quantile(boot_d2, alpha),
                           'D2_hi': np.quantile(boot_d2, 1 - alpha),
            })

    return pd.DataFrame(records)


print('Estimating KM coefficients…')
km_df = estimate_km_bootstrap(
    long_df,
    n_bins=KM_N_BINS, min_count=KM_MIN_COUNT,
    n_boot=KM_N_BOOT, ci=KM_CI,
)
print(f'KM dataframe shape: {km_df.shape}')
print(km_df.groupby('k')[['D1', 'D2']].mean().round(4))

## 6. Pawula diagnostics

The ratio $D^{(4)}/(D^{(2)})^2$ and the excess kurtosis of $\Delta y$
are used to assess the validity of the Fokker–Planck truncation at order 2
(Paper §III.B, Appendix D).

In [ ]:
def compute_higher_order_km(
    long_df: pd.DataFrame,
    n_bins: int = 30,
    min_count: int = 100,
) -> pd.DataFrame:
    """
    Compute D3, D4 and excess kurtosis per transition for Pawula diagnostics
    (Paper §III.B).
    """
    y_global = long_df['y'].values
    q_lo, q_hi = np.nanquantile(y_global, [0.01, 0.99])
    bins = np.linspace(q_lo, q_hi, n_bins + 1)
    records = []

    for k, sub_k in long_df.groupby('k'):
        dx_ref = float(np.nanmedian(sub_k['dx'].values))
        dy_all = sub_k['dy'].values
        dy_all = dy_all[np.isfinite(dy_all)]

        if len(dy_all) < min_count:
            continue

        d2 = np.mean(dy_all ** 2) / (2 * dx_ref)
        d3 = np.mean(dy_all ** 3) / (6 * dx_ref)
        d4 = np.mean(dy_all ** 4) / (24 * dx_ref)

        kurt_excess = (
            np.mean((dy_all - np.mean(dy_all)) ** 4)
            / (np.std(dy_all) ** 4 + 1e-12) - 3
        )
        d4_over_d2sq = d4 / (d2 ** 2 + 1e-12) if d2 != 0 else np.nan

        records.append({
            'k': k,
            'km': SEGMENT_END_KM[k],
            'D2': d2, 'D3': d3, 'D4': d4,
            'D4_over_D2sq': d4_over_d2sq,
            'kurtosis_excess': kurt_excess,
        })

    return pd.DataFrame(records)


km_ho = compute_higher_order_km(long_df)
print('Pawula diagnostics:')
print(f'  Median D4/(D2)²:   {km_ho["D4_over_D2sq"].median():.4f}')
print(f'  Max D4/(D2)²:      {km_ho["D4_over_D2sq"].max():.4f}')
print(f'  Median kurtosis:   {km_ho["kurtosis_excess"].median():.4f}')

## 7. Markov diagnostics

Two tests (Paper §III.C, Appendix D):
1. One-step vs two-step $R^2$ comparison
2. Pearson autocorrelation of $\Delta y$ at lags 1–5

In [ ]:
def compute_markov_diagnostics(
    df: pd.DataFrame, state_cols: list, segment_end_km: np.ndarray
) -> pd.DataFrame:
    """
    Compute one-step vs two-step R² for each transition (Paper §III.C).
    """
    records = []
    for k in range(1, len(state_cols) - 1):
        y_prev = df[state_cols[k - 1]].values
        y_curr = df[state_cols[k]].values
        y_next = df[state_cols[k + 1]].values
        dy     = y_next - y_curr

        mask = np.isfinite(y_prev) & np.isfinite(y_curr) & np.isfinite(dy)
        y_prev, y_curr, dy = y_prev[mask], y_curr[mask], dy[mask]

        # One-step: predict dy from y_curr
        X1 = np.column_stack([np.ones(len(y_curr)), y_curr])
        b1, _, _, _ = np.linalg.lstsq(X1, dy, rcond=None)
        r2_one = r2_score(dy, X1 @ b1)

        # Two-step: predict dy from (y_curr, y_prev)
        X2 = np.column_stack([np.ones(len(y_curr)), y_curr, y_prev])
        b2, _, _, _ = np.linalg.lstsq(X2, dy, rcond=None)
        r2_two = r2_score(dy, X2 @ b2)

        records.append({
            'k': k,
            'km': segment_end_km[k],
            'R2_one': r2_one,
            'R2_two': r2_two,
            'delta_R2': r2_two - r2_one,
        })

    return pd.DataFrame(records)


def compute_autocorrelation(
    df: pd.DataFrame, state_cols: list, max_lag: int = 5
) -> pd.DataFrame:
    """
    Compute Pearson autocorrelation of pacing increments at lags 1–max_lag
    (Paper §III.C).
    """
    # Collect all increments across all runners and transitions
    dy_series = []
    for k in range(len(state_cols) - 1):
        dy = df[state_cols[k + 1]].values - df[state_cols[k]].values
        dy_series.append(dy)
    dy_all = np.concatenate(dy_series)
    dy_all = dy_all[np.isfinite(dy_all)]

    records = []
    for lag in range(1, max_lag + 1):
        x = dy_all[:-lag]
        y = dy_all[lag:]
        r = np.corrcoef(x, y)[0, 1]
        records.append({'lag': lag, 'correlation': r, 'n_pairs': len(x)})

    return pd.DataFrame(records)


markov_df   = compute_markov_diagnostics(df, state_cols, SEGMENT_END_KM)
autocorr_df = compute_autocorrelation(df, state_cols)

print('Markov diagnostics (collapse region k >= 5):')
print(f'  Max ΔR²: {markov_df[markov_df["k"] >= 5]["delta_R2"].max():.4f}')
print()
print('Autocorrelation of Δy:')
print(autocorr_df[['lag', 'correlation']].to_string(index=False))

## 8. First-passage statistics and hazard

The collapse onset distance $\tau_i$ follows Eq. (2) of the paper.
The discrete hazard rate is $h_k = P(\tau = x_k \mid \tau \geq x_k)$.

In [ ]:
# First-passage dataframe from collapse labels
fp_df = df[['wall_persistent', 'wall_km_persistent']].copy()
fp_df.columns = ['event_observed', 'tau_km']
fp_df['event_observed'] = fp_df['event_observed'].astype(int)
fp_df['tau_km'] = fp_df['tau_km'].fillna(SEGMENT_END_KM[-1])

print(f'Collapse rate: {fp_df["event_observed"].mean():.4f}')
print(f'N_wall:        {fp_df["event_observed"].sum():,}')

# Discrete hazard rate
surv_records = []
n_at_risk = len(fp_df)
for km_val in SEGMENT_END_KM:
    events = (fp_df['event_observed'] == 1) & (fp_df['tau_km'] == km_val)
    n_events = events.sum()
    hazard = n_events / n_at_risk if n_at_risk > 0 else 0.0
    surv_records.append({'km': km_val, 'n_events': n_events,
                         'n_at_risk': n_at_risk, 'hazard': hazard})
    n_at_risk -= n_events

surv_all = pd.DataFrame(surv_records)
print('\nHazard rates (x >= 25 km):')
print(surv_all[surv_all['km'] >= 25][['km', 'hazard']].to_string(index=False))

## 9. Effective potential and bifurcation analysis

The effective potential $U(y,x) = -\int^y D^{(1)}(y',x)\,dy'$ (Paper §V, Eq. 5).
The bifurcation distance $x^*$ is identified by the zero crossing of
$D^{(1)}(y{=}0, x)$.

In [ ]:
def smooth_column(df: pd.DataFrame, col: str, sigma: float = 1.0) -> pd.DataFrame:
    """Apply Gaussian smoothing to a column."""
    df = df.copy()
    df[f'{col}_smooth'] = gaussian_filter1d(df[col].values, sigma=sigma)
    return df


# Effective potential per transition
potential_dict = {}
for kk in sorted(km_df['k'].unique()):
    sub = km_df[km_df['k'] == kk].sort_values('state_center').copy()
    if len(sub) < 3:
        continue
    sub = smooth_column(sub, 'D1')
    y   = sub['state_center'].values
    d1s = sub['D1_smooth'].values
    U   = -cumulative_trapezoid(d1s, y, initial=0)
    U  -= U.min()
    sub['U'] = U
    potential_dict[kk] = sub

# D1 evaluated at origin y=0 per transition
d1_at_zero = []
for kk in sorted(km_df['k'].unique()):
    sub = km_df[km_df['k'] == kk].sort_values('state_center')
    if len(sub) < 3:
        continue
    f = interp1d(sub['state_center'], sub['D1'],
                 kind='linear', fill_value='extrapolate')
    d1_at_zero.append({'k': kk, 'km': SEGMENT_END_KM[kk],
                       'D1_at_y0': float(f(0.0))})

d1_zero_df = pd.DataFrame(d1_at_zero)
print('D1(y=0, x):')
print(d1_zero_df.round(4).to_string(index=False))

# Bifurcation distance via linear interpolation of zero crossing
neg_mask = d1_zero_df['D1_at_y0'] < 0
pos_mask = d1_zero_df['D1_at_y0'] > 0
km1 = d1_zero_df.loc[neg_mask, 'km'].iloc[-1]
km2 = d1_zero_df.loc[pos_mask, 'km'].iloc[0]
d1  = d1_zero_df.loc[neg_mask, 'D1_at_y0'].iloc[-1]
d2  = d1_zero_df.loc[pos_mask, 'D1_at_y0'].iloc[0]
km_bifurcation = km1 + (0 - d1) * (km2 - km1) / (d2 - d1)
print(f'\nBifurcation distance x* = {km_bifurcation:.2f} km')

## 10. Langevin simulation — in-sample validation

Euler–Maruyama integration of Eq. (6) (Paper §VI.A).

In [ ]:
def build_interpolators(
    km_df: pd.DataFrame, min_points: int = 5
) -> dict:
    """
    Build drift and diffusion interpolators per transition k.

    Returns
    -------
    dict: {k: {'D1': interp1d, 'D2': interp1d}}
    """
    interps = {}
    for kk in sorted(km_df['k'].unique()):
        sub = km_df[km_df['k'] == kk].sort_values('state_center').copy()
        if len(sub) < min_points:
            continue
        sub = smooth_column(sub, 'D1')
        sub = smooth_column(sub, 'D2')
        x   = sub['state_center'].values
        interps[kk] = {
            'D1': interp1d(x, sub['D1_smooth'].values, kind='linear',
                           fill_value='extrapolate', bounds_error=False),
            'D2': interp1d(x, np.clip(sub['D2_smooth'].values, 1e-8, None),
                           kind='linear', fill_value='extrapolate',
                           bounds_error=False),
        }
    return interps


def simulate_paths(
    interp_dict: dict,
    n_paths: int = 30_000,
    y0_source: np.ndarray = None,
    clip_low: float = -0.5,
    clip_high: float = 1.5,
    seed: int = 42,
) -> np.ndarray:
    """
    Euler–Maruyama integration of the Langevin equation (Paper §VI.A, Eq. 7).

    Returns
    -------
    np.ndarray of shape (n_paths, n_segments)
    """
    rng   = np.random.default_rng(seed=seed)
    n_seg = len(state_cols)
    paths = np.full((n_paths, n_seg), np.nan)

    if y0_source is None:
        paths[:, 0] = np.zeros(n_paths)
    else:
        paths[:, 0] = np.asarray(y0_source)

    for k in range(n_seg - 1):
        if k not in interp_dict:
            paths[:, k + 1] = paths[:, k]
            continue
        dx  = NORM_DIST[k + 1] - NORM_DIST[k]
        yk  = np.clip(paths[:, k], clip_low, clip_high)
        d1  = interp_dict[k]['D1'](yk)
        d2  = np.clip(interp_dict[k]['D2'](yk), 1e-8, None)
        xi  = rng.standard_normal(n_paths)
        paths[:, k + 1] = np.clip(
            yk + d1 * dx + np.sqrt(2 * d2 * dx) * xi,
            clip_low, clip_high
        )
    return paths


def detect_wall_from_paths(
    paths: np.ndarray,
    threshold: float = 0.10,
    start_idx: int = 5,
) -> pd.DataFrame:
    """
    Detect persistent-wall events in simulated trajectories (Paper §II.C).

    Returns
    -------
    pd.DataFrame with columns: event_observed, tau_km
    """
    records = []
    for path in paths:
        event, tau_km = 0, np.nan
        for idx in range(start_idx, paths.shape[1] - 1):
            if path[idx] > threshold and path[idx + 1] > threshold:
                event  = 1
                tau_km = float(SEGMENT_END_KM[idx])
                break
        records.append({'event_observed': event, 'tau_km': tau_km})

    result = pd.DataFrame(records)
    result['tau_km'] = result['tau_km'].fillna(SEGMENT_END_KM[-1])
    return result


# In-sample simulation
print('Building interpolators…')
interp_dict = build_interpolators(km_df)

print(f'Simulating {SIM_N_PATHS:,} paths (in-sample)…')
y0 = df[state_cols[0]].sample(SIM_N_PATHS, replace=True,
                               random_state=42).values
sim_paths = simulate_paths(interp_dict, n_paths=SIM_N_PATHS, y0_source=y0)
sim_fp    = detect_wall_from_paths(sim_paths)

emp_mean = df[state_cols].mean().values
sim_mean = np.nanmean(sim_paths, axis=0)
r2_mean  = r2_score(emp_mean, sim_mean)

emp_tau = fp_df.loc[fp_df['event_observed'] == 1, 'tau_km'].values
sim_tau = sim_fp.loc[sim_fp['event_observed'] == 1, 'tau_km'].values

ks_stat, ks_pval = ks_2samp(emp_tau, sim_tau)
w1 = wasserstein_distance(emp_tau, sim_tau)

print(f'\nIn-sample results:')
print(f'  R² (mean trajectory): {r2_mean:.4f}')
print(f'  Wall rate empirical:  {fp_df["event_observed"].mean():.4f}')
print(f'  Wall rate simulated:  {sim_fp["event_observed"].mean():.4f}')
print(f'  KS statistic:         {ks_stat:.4f}')
print(f'  Wasserstein W1 (km):  {w1:.4f}')

## 11. Out-of-sample validation (Berlin Marathon)

KM coefficients estimated on training set (Boston, Chicago, New York);
simulations applied to the held-out Berlin dataset (Paper §VI.B).

In [ ]:
df_full = df.copy()  # full dataset including Berlin

if 'City' not in df_full.columns:
    print('WARNING: No City column — OOS validation skipped.')
    CROSSVAL_AVAILABLE = False
else:
    train_mask = df_full['City'].isin(TRAIN_CITIES)
    test_mask  = df_full['City'].isin(TEST_CITIES)

    df_train = df_full[train_mask].reset_index(drop=True)
    df_test  = df_full[test_mask].reset_index(drop=True)

    print(f'Train: {len(df_train):,} runners ({TRAIN_CITIES})')
    print(f'Test:  {len(df_test):,} runners  ({TEST_CITIES})')

    # Build training transition dataframe and estimate KM
    long_train = build_transition_df(
        df_train, state_cols, NORM_DIST, SEGMENT_END_KM
    )
    print('Estimating KM on training set…')
    km_train = estimate_km_bootstrap(
        long_train, n_bins=28, min_count=150, n_boot=200
    )

    # Simulate on test set
    interp_train = build_interpolators(km_train)
    y0_test = df_test[state_cols[0]].sample(
        20_000, replace=True, random_state=99
    ).values
    sim_paths_oos = simulate_paths(
        interp_train, n_paths=20_000, y0_source=y0_test, seed=99
    )
    sim_fp_oos = detect_wall_from_paths(sim_paths_oos)

    emp_mean_test = df_test[state_cols].mean().values
    sim_mean_oos  = np.nanmean(sim_paths_oos, axis=0)
    r2_oos = r2_score(emp_mean_test, sim_mean_oos)

    fp_berlin   = detect_wall_from_paths(
        df_test[state_cols].values,
        threshold=Y_THRESHOLD, start_idx=LATE_START_IDX
    )
    emp_tau_oos = fp_berlin.loc[fp_berlin['event_observed'] == 1, 'tau_km'].values
    sim_tau_oos = sim_fp_oos.loc[sim_fp_oos['event_observed'] == 1, 'tau_km'].values
    ks_oos, p_oos = ks_2samp(emp_tau_oos, sim_tau_oos)
    w1_oos = wasserstein_distance(emp_tau_oos, sim_tau_oos)

    print(f'\nOOS results (Berlin):')
    print(f'  R² (mean trajectory): {r2_oos:.4f}')
    print(f'  Wall rate empirical:  {df_test["wall_persistent"].mean():.4f}')
    print(f'  Wall rate simulated:  {sim_fp_oos["event_observed"].mean():.4f}')
    print(f'  KS statistic:         {ks_oos:.4f}')
    print(f'  Wasserstein W1 (km):  {w1_oos:.4f}')

    CROSSVAL_AVAILABLE = True

## 12. Robustness checks

Sensitivity of the collapse rate to the threshold $y_c$ (Appendix A)
and to the state variable definition (Appendix B).

In [ ]:
# Threshold robustness (Appendix A)
threshold_values = [0.08, 0.10, 0.12, 0.15]
robust_thr_records = []
for yc in threshold_values:
    wall_any = False
    n_wall_yc = 0
    y_mat = df[state_cols].values
    for i in range(len(df)):
        for k in range(LATE_START_IDX, len(state_cols) - 1):
            if y_mat[i, k] > yc and y_mat[i, k + 1] > yc:
                n_wall_yc += 1
                break
    robust_thr_records.append({
        'threshold': yc,
        'wall_rate': n_wall_yc / len(df)
    })

robust_thr = pd.DataFrame(robust_thr_records)
print('Threshold robustness:')
print(robust_thr.to_string(index=False))

## 13. Publication figures (PRE style)

In [ ]:
# ── Figure 1: Mean pacing trajectory ─────────────────────────────────────────
q10       = state_matrix.quantile(0.10).values
q90       = state_matrix.quantile(0.90).values
mean_traj = state_matrix.mean().values

fig, ax = plt.subplots(figsize=(5.2, 4.0))
ax.fill_between(SEGMENT_MID_KM, q10, q90,
                color='0.78', label='10–90 percentile')
ax.plot(SEGMENT_MID_KM, mean_traj,
        color='black', marker='o', markersize=5,
        linewidth=2, label=r'$\langle y(x)\rangle$')
ax.axhline(0, linestyle='--', linewidth=1, color='0.5')
ax.set_xlabel(r'Distance $x$ (km)')
ax.set_ylabel(r'Mean pacing deviation $\langle y(x)\rangle$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig1_mean_pacing.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure 2: Drift and diffusion with bootstrap CI ───────────────────────────
REPRE_K   = [0, 4, 8]
REPRE_LAB = ['0–5 km', '20–21.1 km', '35–40 km']
MARKERS   = ['o', 's', '^']
COLORS    = ['black', '0.45', '0.72']

x_min = km_df['state_center'].min()
x_max = km_df['state_center'].max()

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))

for ax_idx, coef, ylabel in zip(
        [0, 1],
        ['D1', 'D2'],
        [r'Drift $D^{(1)}(y,x)$', r'Diffusion $D^{(2)}(y,x)$']):
    ax = axes[ax_idx]

    # Collapse region — solid rectangle (no alpha for EPS compatibility)
    ax.axvspan(Y_THRESHOLD, x_max, color='0.93')
    ax.axvline(Y_THRESHOLD, linestyle=':', color='0.5', linewidth=1.2)

    for kk, lbl, mk, col in zip(REPRE_K, REPRE_LAB, MARKERS, COLORS):
        sub = km_df[km_df['k'] == kk].sort_values('state_center')
        if sub.empty:
            continue
        ax.errorbar(
            sub['state_center'], sub[coef],
            yerr=[sub[coef] - sub[f'{coef}_lo'],
                  sub[f'{coef}_hi'] - sub[coef]],
            marker=mk, markersize=6, color=col, linewidth=1.8,
            elinewidth=0.9, capsize=3, label=lbl
        )

    if ax_idx == 0:
        ax.axhline(0, linestyle='--', color='0.5', linewidth=1)
        ax.axvline(Y_THRESHOLD, linestyle=':', color='0.5', linewidth=1.2,
                   label=rf'$y_c={Y_THRESHOLD}$')

    ax.text(0.72, 0.92, 'Collapse\nregion',
            transform=ax.transAxes, fontsize=9, color='0.35',
            ha='center', style='italic', va='top')
    ax.set_xlabel(r'State $y$')
    ax.set_ylabel(ylabel)
    loc = 'lower left' if ax_idx == 0 else 'upper left'
    ax.legend(frameon=True, edgecolor='0.8', fancybox=False,
              fontsize=11, loc=loc)
    label = '(a)' if ax_idx == 0 else '(b)'
    ax.text(0.88, 0.92, label, transform=ax.transAxes,
            fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig2_drift_diffusion_PRE_CI.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure 3: First-passage distribution and hazard ───────────────────────────
obs_tau = fp_df.loc[fp_df['event_observed'] == 1, 'tau_km'].values
bins    = np.array([22.5, 27.5, 32.5, 37.5, 42.5])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))

# Panel (a): P(tau)
ax = axes[0]
ax.hist(obs_tau, bins=bins, density=True, histtype='stepfilled',
        color='0.85', edgecolor='0.5', linewidth=1.5)
x_grid = np.linspace(22, 43, 400)
ax.plot(x_grid, gaussian_kde(obs_tau)(x_grid), color='black', linewidth=2.2)
ax.set_xlim(22, 43)
ax.set_xlabel(r'First-passage distance $\tau$ (km)')
ax.set_ylabel(r'Probability density $P(\tau)$')
ax.text(0.88, 0.92, '(a)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

# Panel (b): hazard rate
ax = axes[1]
surv_late = surv_all[surv_all['km'] >= 25].copy()
bars = ax.bar(surv_late['km'], surv_late['hazard'],
              width=4.0, color='0.75', edgecolor='black', linewidth=1.0)

# Highlight maximum hazard bar
max_idx = surv_late['hazard'].idxmax()
bars[surv_late.index.get_loc(max_idx)].set_facecolor('0.3')

for _, row in surv_late.iterrows():
    if row['hazard'] > 0:
        ax.text(row['km'], row['hazard'] + 0.005,
                f"{row['hazard']:.2f}", ha='center', va='bottom', fontsize=10)

ax.set_xlabel(r'Distance $x$ (km)')
ax.set_ylabel(r'Hazard rate $h_k$')
ax.set_ylim(0, surv_late['hazard'].max() * 1.25)
ax.text(0.88, 0.92, '(b)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig3_first_passage_hazard_PRE.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure 4: Effective potential and bifurcation ─────────────────────────────
PLOT_K   = [0, 4, 8]
PLOT_LAB = ['5–10 km', '20–21.1 km', '35–40 km']

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0))

# Panel (a): effective potential
ax = axes[0]
for kk, lbl, mk, col in zip(PLOT_K, PLOT_LAB, MARKERS, COLORS):
    if kk not in potential_dict:
        continue
    sub = potential_dict[kk]
    ax.plot(sub['state_center'], sub['U'],
            marker=mk, markersize=5, color=col, linewidth=1.8, label=lbl)
ax.set_xlabel(r'State $y$')
ax.set_ylabel(r'Effective potential $U(y,x)$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.88, 0.10, '(a)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

# Panel (b): D1(y=0) vs distance
ax = axes[1]
ax.axvspan(km_bifurcation, 41, color='0.93')   # no alpha for EPS
ax.plot(d1_zero_df['km'], d1_zero_df['D1_at_y0'],
        marker='o', color='black', linewidth=2, markersize=7)
ax.axhline(0, linestyle='--', color='0.5', linewidth=1)
ax.axvline(km_bifurcation, linestyle=':', color='black', linewidth=1.5,
           label=f'Bifurcation ($x^*={km_bifurcation:.1f}$ km)')

ax.text(0.62, 0.88, 'Unstable\n(no attractor)',
        transform=ax.transAxes, fontsize=10, color='0.35',
        ha='center', style='italic', va='top')
ax.text(0.12, 0.45, 'Stable',
        transform=ax.transAxes, fontsize=10, color='0.35',
        ha='center', style='italic')
ax.annotate('Survivor\nbias',
            xy=(40, -0.42), xytext=(37.5, -0.22),
            fontsize=9, color='0.35', style='italic',
            arrowprops=dict(arrowstyle='->', color='0.5', lw=0.8))

ax.set_xlabel(r'Distance $x$ (km)')
ax.set_ylabel(r'$D^{(1)}(y{=}0,\, x)$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11,
          loc='lower left')
ax.text(0.88, 0.92, '(b)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig4_potential_bifurcation_PRE.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure 5: In-sample simulation validation ─────────────────────────────────
emp_tau_is = fp_df.loc[fp_df['event_observed'] == 1, 'tau_km'].values
sim_tau_is = sim_fp.loc[sim_fp['event_observed'] == 1, 'tau_km'].values

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))

# Panel (a): mean trajectory
ax = axes[0]
ax.plot(SEGMENT_MID_KM, emp_mean,
        marker='o', color='black', linewidth=2, markersize=7, label='Empirical')
ax.plot(SEGMENT_MID_KM, sim_mean,
        marker='s', color='0.45', linewidth=2, markersize=7,
        linestyle='--', label='Simulated')
ax.axhline(0, linestyle=':', color='0.6', linewidth=1)
ax.text(0.58, 0.10, rf'$R^2={r2_mean:.2f}$',
        transform=ax.transAxes, fontsize=14)
ax.set_xlabel('Distance (km)')
ax.set_ylabel(r'$\langle y(x)\rangle$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.88, 0.92, '(a)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

# Panel (b): first-passage distributions
ax = axes[1]
bins = np.array([22.5, 27.5, 32.5, 37.5, 42.5])
ax.hist(emp_tau_is, bins=bins, density=True, histtype='stepfilled',
        color='0.85', edgecolor='black', linewidth=1.5, label='Empirical')
ax.hist(sim_tau_is, bins=bins, density=True, histtype='step',
        color='0.45', linewidth=2.2, linestyle='--', label='Simulated')
ax.plot(np.linspace(22, 43, 400),
        gaussian_kde(emp_tau_is, bw_method=0.3)(np.linspace(22, 43, 400)),
        color='black', linewidth=1.8)
ax.plot(np.linspace(22, 43, 400),
        gaussian_kde(sim_tau_is, bw_method=0.3)(np.linspace(22, 43, 400)),
        color='0.45', linewidth=1.8, linestyle='--')
ax.text(0.35, 0.88, rf'$W_1={w1:.2f}$ km',
        transform=ax.transAxes, fontsize=12)
ax.set_xlim(22, 43)
ax.set_xlabel(r'First-passage distance $\tau$ (km)')
ax.set_ylabel(r'Probability density $P(\tau)$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.88, 0.92, '(b)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig5_simulation_validation_PRE.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure 6 (Appendix A): Threshold robustness ───────────────────────────────
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.plot(robust_thr['threshold'], robust_thr['wall_rate'],
        marker='o', color='black', linewidth=2, markersize=7)

for i, (_, row) in enumerate(robust_thr.iterrows()):
    offset = -0.012 if i == 0 else 0.006
    va     = 'top'  if i == 0 else 'bottom'
    ax.text(row['threshold'], row['wall_rate'] + offset,
            f"{row['wall_rate']:.3f}", ha='center', va=va, fontsize=10)

ax.axvline(Y_THRESHOLD, linestyle=':', color='0.5', linewidth=1.2,
           label=f'Primary threshold ($y_c={Y_THRESHOLD}$)')
ax.set_xlabel(r'Threshold $y_c$')
ax.set_ylabel('Persistent-wall rate')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig6_app_threshold_robustness.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure A (Appendix D): Pawula diagnostics ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

ax = axes[0]
ax.plot(km_ho['km'], km_ho['D4_over_D2sq'],
        marker='o', color='black', linewidth=1.8, markersize=6)
ax.axhline(0.5, linestyle=':', color='0.5', linewidth=1.2,
           label='Threshold (0.5)')
ax.set_xlabel('Distance (km)')
ax.set_ylabel(r'$D^{(4)}/(D^{(2)})^2$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.05, 0.92, '(a)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

ax = axes[1]
ax.plot(km_ho['km'], km_ho['kurtosis_excess'],
        marker='o', color='black', linewidth=1.8, markersize=6)
ax.axhline(0, linestyle='--', color='0.5', linewidth=1)
ax.set_xlabel('Distance (km)')
ax.set_ylabel(r'Excess kurtosis of $\Delta y$')
ax.text(0.05, 0.92, '(b)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'figA_pawula.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure B (Appendix D): Markov diagnostics ────────────────────────────────
from matplotlib.patches import Rectangle

lag1_corr = autocorr_df.loc[autocorr_df['lag'] == 1, 'correlation'].values[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

# Panel (a)
ax = axes[0]
ax.add_patch(Rectangle(
    (25, -0.05), 42.5 - 25, 1.1,
    facecolor='0.93', edgecolor='none', zorder=0
))
ax.plot(markov_df['km'], markov_df['R2_one'],
        marker='o', label='One-step', color='black',
        linewidth=1.8, markersize=6, zorder=2)
ax.plot(markov_df['km'], markov_df['R2_two'],
        marker='s', label='Two-step', color='0.5',
        linestyle='--', linewidth=1.5, markersize=6, zorder=2)
ax.text(0.72, 0.88, 'Collapse\nregion',
        transform=ax.transAxes, fontsize=10, color='0.35',
        ha='center', style='italic', va='top')
ax.set_xlabel('Distance (km)')
ax.set_ylabel(r'$R^2$')
ax.set_xlim(8, 43)
ax.set_ylim(-0.05, 0.85)
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.05, 0.92, '(a)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

# Panel (b)
ax = axes[1]
ax.bar(autocorr_df['lag'], autocorr_df['correlation'],
       color='0.35', width=0.5, zorder=2)
ax.axhline(0, color='0.5', linewidth=1, zorder=1)
n_eff   = int(autocorr_df['n_pairs'].mean())
ci_band = 1.96 / np.sqrt(n_eff)
ax.axhline( ci_band, color='0.5', linewidth=1, linestyle='--',
            label='95% CI (zero corr.)', zorder=1)
ax.axhline(-ci_band, color='0.5', linewidth=1, linestyle='--', zorder=1)
ax.set_ylim(-0.25, 0.05)
ax.annotate(f'$r = {lag1_corr:.3f}$', xy=(1, lag1_corr),
            xytext=(1.5, lag1_corr - 0.025), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
ax.set_xlabel('Lag (segments)')
ax.set_ylabel(r'$\mathrm{Corr}(\Delta y_k,\, \Delta y_{k+\ell})$')
ax.legend(frameon=True, edgecolor='0.8', fancybox=False, fontsize=11)
ax.text(0.05, 0.92, '(b)', transform=ax.transAxes,
        fontsize=15, fontweight='bold')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'figB_markov.eps', dpi=300)
plt.show()

In [ ]:
# ── Figure C (Appendix E): OOS validation — Berlin Marathon ──────────────────
if CROSSVAL_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))

    ax = axes[0]
    ax.plot(SEGMENT_MID_KM, emp_mean_test,
            marker='o', color='black', linewidth=2, markersize=7,
            label='Empirical (Berlin)')
    ax.plot(SEGMENT_MID_KM, sim_mean_oos,
            marker='s', color='0.45', linewidth=2, markersize=7,
            linestyle='--', label='Simulated (train→Berlin)')
    ax.axhline(0, linestyle=':', color='0.6', linewidth=1)
    ax.text(0.58, 0.10, rf'$R^2={r2_oos:.2f}$',
            transform=ax.transAxes, fontsize=14)
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel(r'$\langle y(x)\rangle$')
    ax.legend(frameon=True, edgecolor='0.8', fancybox=False)
    ax.text(0.88, 0.92, '(a)', transform=ax.transAxes,
            fontsize=15, fontweight='bold')

    ax = axes[1]
    bins = np.array([22.5, 27.5, 32.5, 37.5, 42.5])
    ax.hist(emp_tau_oos, bins=bins, density=True,
            histtype='step', color='black', linewidth=2.2,
            label='Empirical (Berlin)')
    ax.hist(sim_tau_oos, bins=bins, density=True,
            histtype='step', color='0.5', linewidth=2.2,
            linestyle='--', label='Simulated OOS')
    ax.set_xlabel(r'First-passage distance $\tau$ (km)')
    ax.set_ylabel(r'$P(\tau)$')
    ax.legend(frameon=True, edgecolor='0.8', fancybox=False)
    ax.text(0.88, 0.92, '(b)', transform=ax.transAxes,
            fontsize=15, fontweight='bold')

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'figC_out_of_sample.eps', dpi=300)
    plt.show()

## 14. Diagnostic summary

In [ ]:
summary = pd.DataFrame([
    # Dataset
    ('Total runners',                        f'{len(df):,}'),
    ('Segments',                             str(len(state_cols))),
    ('Marathons',                            str(df['City'].nunique())),
    # Markov
    ('Max ΔR² (Markov, full)',               f'{markov_df["delta_R2"].max():.5f}'),
    ('Max ΔR² (collapse region, k≥5)',       f'{markov_df[markov_df["k"]>=5]["delta_R2"].max():.5f}'),
    ('Autocorr Δy lag-1',                    f'{autocorr_df.loc[autocorr_df["lag"]==1, "correlation"].values[0]:.5f}'),
    # Pawula
    ('Median D4/(D2)²',                      f'{km_ho["D4_over_D2sq"].median():.4f}'),
    ('Max D4/(D2)²',                         f'{km_ho["D4_over_D2sq"].max():.4f}'),
    ('Median kurtosis excess',               f'{km_ho["kurtosis_excess"].median():.4f}'),
    # KM estimation
    ('KM bins',                              str(KM_N_BINS)),
    ('Bootstrap resamples',                  str(KM_N_BOOT)),
    ('Min count per bin',                    str(KM_MIN_COUNT)),
    # Bifurcation
    ('Bifurcation x* (km)',                  f'{km_bifurcation:.2f}'),
    # First-passage
    ('Empirical wall rate',                  f'{fp_df["event_observed"].mean():.4f}'),
    ('Wall events',                          f'{fp_df["event_observed"].sum():,}'),
    ('Hazard peak (km)',                     f'{surv_all.loc[surv_all["hazard"].idxmax(), "km"]:.1f}'),
    # In-sample
    ('R² mean trajectory (in-sample)',       f'{r2_mean:.4f}'),
    ('Simulated wall rate (in-sample)',      f'{sim_fp["event_observed"].mean():.4f}'),
    ('KS statistic (in-sample)',             f'{ks_stat:.4f}'),
    ('Wasserstein W1 in-sample (km)',        f'{w1:.4f}'),
    # OOS
    ('R² mean trajectory (OOS Berlin)',      f'{r2_oos:.4f}'   if CROSSVAL_AVAILABLE else 'N/A'),
    ('KS statistic (OOS Berlin)',            f'{ks_oos:.4f}'   if CROSSVAL_AVAILABLE else 'N/A'),
    ('Wasserstein W1 OOS (km)',              f'{w1_oos:.4f}'   if CROSSVAL_AVAILABLE else 'N/A'),
    ('Simulated wall rate (OOS Berlin)',     f'{sim_fp_oos["event_observed"].mean():.4f}'
                                             if CROSSVAL_AVAILABLE else 'N/A'),
], columns=['Metric', 'Value'])

print('=== PRE SUBMISSION DIAGNOSTIC SUMMARY ===')
print(summary.to_string(index=False))
summary.to_csv(RESULTS_DIR / 'PRE_diagnostic_summary.csv', index=False)
print('\nSaved to results/PRE_diagnostic_summary.csv')



### Diagnostics
- [ ] Pawula: median $D^{(4)}/(D^{(2)})^2 < 0.5$ at all transitions
- [ ] Markov: max $\Delta R^2 \leq 0.034$ in the collapse region ($k \geq 5$)
- [ ] Bifurcation confirmed by zero crossing of $D^{(1)}(y{=}0,x)$

### Validation
- [ ] $R^2 \geq 0.88$ in-sample (mean trajectory)
- [ ] $R^2 \geq 0.64$ out-of-sample (Berlin)
- [ ] $W_1 < 2.5$ km (in-sample and OOS)

### Figures (all .EPS)
- [ ] `fig1_mean_pacing.eps`
- [ ] `fig2_drift_diffusion_PRE_CI.eps`
- [ ] `fig3_first_passage_hazard_PRE.eps`
- [ ] `fig4_potential_bifurcation_PRE.eps`
- [ ] `fig5_simulation_validation_PRE.eps`
- [ ] `fig6_app_threshold_robustness.eps`
- [ ] `figA_pawula.eps`
- [ ] `figB_markov.eps`
- [ ] `figC_out_of_sample.eps`